In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/ml2022spring-hw1/covid.test.csv
/kaggle/input/competitions/ml2022spring-hw1/covid.train.csv


In [2]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import TensorDataset,DataLoader

In [3]:
train_data=pd.read_csv("/kaggle/input/competitions/ml2022spring-hw1/covid.train.csv")
train_data.shape
x1=train_data.iloc[:,53:58]
x2=train_data.iloc[:,69:74]
x3=train_data.iloc[:,85:90]
x4=train_data.iloc[:,101:106]


new_index=["tested_positive","cli","ili","hh_cmnty_cli","nohh_cmnty_cli"]
x1.columns=[new_index]
x2.columns=[new_index]
x3.columns=[new_index]
x4.columns=[new_index]
x=pd.concat([x1,x2,x3,x4],axis=0)
x

,tested_positive,cli,ili,hh_cmnty_cli,nohh_cmnty_cli
0,7.374846,0.653157,0.713249,12.488933,8.219380
1,9.850027,0.738029,0.720511,15.070049,10.990937
2,3.897851,0.663597,0.635459,8.472831,5.154184
3,10.297712,1.356577,1.432775,12.997184,7.843776
4,24.056627,1.298511,1.231606,20.189998,16.229791
...,...,...,...,...,...
2694,12.962922,1.417447,1.401417,26.355591,21.616363
2695,5.032925,0.605297,0.608920,11.135534,8.133172
2696,14.982000,1.477089,1.566401,27.571493,21.874254
2697,8.911676,0.829967,0.825078,14.424899,11.189104


In [4]:
y1=train_data.iloc[:,53:54]
y2=train_data.iloc[:,69:70]
y3=train_data.iloc[:,85:86]
y4=train_data.iloc[:,101:102]

y1.columns=["test_positives"]
y2.columns=["test_positives"]
y3.columns=["test_positives"]
y4.columns=["test_positives"]


y=pd.concat([y1,y2,y3,y4],axis=0)
y

,test_positives
0,7.374846
1,9.850027
2,3.897851
3,10.297712
4,24.056627
...,...
2694,12.962922
2695,5.032925
2696,14.982000
2697,8.911676


In [5]:
x_np=x.to_numpy()
y_np=y.to_numpy()
#在numpy下使用标准化一下，不然loss无法下降
x_min=x_np.min()
x_max=x_np.max()
x_np=(x_np-x_min)/(x_max-x_min)

#y_min=y_np.min()
#y_max=y_np.max()
#y_np=(y_np-y_min)/(y_max-y_min)

x_train=torch.tensor(x_np, dtype=torch.float32)
y_train=torch.tensor(y_np,dtype=torch.float32)


train_dataset=TensorDataset(x_train,y_train)
train_loader=DataLoader(train_dataset,batch_size=1000,shuffle=True)

In [6]:

class LinearRegression(torch.nn.Module):
    def __init__(self):
        super(LinearRegression,self).__init__()

        self.layer=torch.nn.Linear(5,1)
    def forward(self,x):
        y_pred=self.layer(x)
        return y_pred

model=LinearRegression()
critrision=torch.nn.MSELoss(reduction="mean")
optimizer=torch.optim.SGD(model.parameters(),lr=0.1)

model.train()
for i in range(1001):
    for batch_x,batch_y in train_loader:
        y_pred=model(batch_x)
        loss=critrision(y_pred,batch_y)
        if(i%200==0):
            print(i,loss.item())
    
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

0 147.93321228027344
0 86.31185913085938
0 53.57387161254883
0 45.30329132080078
0 35.05948257446289
0 30.33413314819336
0 28.037437438964844
0 23.55923080444336
0 23.27956199645996
0 23.282636642456055
0 22.46491241455078
200 0.18096789717674255
200 0.18992379307746887
200 0.17757761478424072
200 0.19481153786182404
200 0.2093309760093689
200 0.19277100265026093
200 0.18954914808273315
200 0.19476774334907532
200 0.1805552840232849
200 0.20987869799137115
200 0.18779781460762024
400 0.0067426166497170925
400 0.0068388767540454865
400 0.007269653957337141
400 0.007425881456583738
400 0.00781854335218668
400 0.00766067486256361
400 0.007430264260619879
400 0.007352311629801989
400 0.007500730454921722
400 0.00835502240806818
400 0.008133835159242153
600 0.0004588206356856972
600 0.0004099195066373795
600 0.0004316122503951192
600 0.0004623111162800342
600 0.0004634144133888185
600 0.000436050962889567
600 0.0004229439073242247
600 0.0004895818419754505
600 0.00043143992661498487
600 0.0

In [7]:
data_test=pd.read_csv("/kaggle/input/competitions/ml2022spring-hw1/covid.test.csv")
x_pd=data_test.iloc[:,101:106]
xt_np=x_pd.to_numpy()
xt_np=(xt_np-x_min)/(x_max-x_min)

x_test=torch.tensor(xt_np,dtype=torch.float32)           


In [8]:
test_ids = data_test.iloc[:, 0].values
model.eval() # 设置为评估模式，这是一个好习惯
with torch.no_grad(): # 关闭梯度计算，节省内存，加速推理
    y_pred_tensor = model(x_test)

# 将结果转为 numpy 数组，并压平成一维
# 模型输出是 [[1.2], [3.4]]，需要变成 [1.2, 3.4]
y_pred = y_pred_tensor.numpy().flatten()

# ==========================================
# 4. 生成并提交 CSV 文件
# ==========================================
# 创建一个 DataFrame，列名必须和 sample_submission.csv 中的一致
submission = pd.DataFrame({
    "id": test_ids,
    "tested_positive": y_pred 
})

# 保存为 csv 文件，index=False 表示不保存行号
submission.to_csv("submission.csv", index=False)

print("✅ 提交文件 submission.csv 已成功生成！")
print("文件前 5 行预览：")
print(submission.head())

✅ 提交文件 submission.csv 已成功生成！
文件前 5 行预览：
   id  tested_positive
0   0         8.355362
1   1         9.305448
2   2         5.032653
3   3         8.973656
4   4        17.334942
